# 1. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# 2. Read Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

# 3. Silver Layer Transformations

## 3.1. Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## 3.2. Customer ID Cleanup

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

In [0]:
df.limit(10).display()

## 3.3. Validate Bithdate
Set future birthdates to Null

Rule: `birth_date` < Current Date 



In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

In [0]:
# Data Quality Checks
df.limit(10).display()

## 3.4. Gender Column Normalization

In [0]:
df = (
    df
    .withColumn(
        "gen",
        F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
        .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
        .otherwise("n/a")
    )
)

## 3.5. Renaming Columns

In [0]:

RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
# Sanity Checks of Dataframe
df.limit(10).display()

# 4. Writing Table To Silver Layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

In [0]:
%sql
-- Sanity Checks of Silver Table
SELECT * FROM workspace.silver.erp_customers LIMIT 10